In [9]:
from pathlib import Path
import json
import re


def load_benchmark_files(
    directory="benchmarks",
    throughput=None,
    n_scans_per_batch=None,
    n_workers=None,
    n_partitions=None,
    verify_content=True,
):
    """
    Load benchmark JSON files matching requested parameters.

    Parameters
    ----------
    directory : str
        Benchmark directory.
    throughput : int | None
        Required throughput in MBps. None means any.
    n_scans_per_batch : int | None
        Required scans per batch. None means any.
    n_partitions : int | None
        Required partition count. None means any.
    verify_content : bool
        If True, verify filename metadata matches JSON content.

    Returns
    -------
    list[dict]
        List of dicts with:
        {
            "path": Path,
            "throughput": int,
            "spb": int,
            "parts": int,
            "data": dict
        }
    """

    # Build glob pattern
    throughput_pattern = f"{throughput}MBps" if throughput is not None else "*MBps"
    
    nspb_pattern = f"{n_scans_per_batch}SpB" if n_scans_per_batch is not None else "*SpB"
    nparts_pattern = f"{n_partitions}parts" if n_partitions is not None else "*parts"
    nws_pattern = f"{n_workers}ws" if n_workers is not None else "*ws"

    pattern = f"*--{throughput_pattern}--{nspb_pattern}--{nws_pattern}--{nparts_pattern}.json"

    filename_re = re.compile(
        r".+--(?P<throughput>\d+)MBps--"
        r"(?P<spb>\d+)SpB--"
        r"(?P<ws>\d+)ws--"
        r"(?P<parts>\d+)parts\.json$"
    )

    results = []

    for path in Path(directory).glob(pattern):
        match = filename_re.match(path.name)
        if not match:
            continue

        filename_values = {
            "throughput_MB": int(match.group("throughput")),
            "n_scans_per_batch": int(match.group("spb")),
            "n_workers": int(match.group("ws")),
            "n_partitions": int(match.group("parts")),
        }

        with path.open() as f:
            content = json.load(f)

            # Adjust these keys if your JSON uses different names
            content_values = {
                "throughput_MB": content["throughput_MB"],
                "n_scans_per_batch": content["n_scans_per_batch"],
                "n_workers": content["n_workers"],
                "n_partitions": content["n_partitions"],
            }

            if filename_values != content_values:
                raise ValueError(
                    f"Filename/content mismatch:\n"
                    f"File: {path}\n"
                    f"Filename values: {filename_values}\n"
                    f"JSON values: {content_values}"
                )

            results.append(
                {
                    "path": path,
                    **filename_values,
                    "data": content,
                })
    return results

In [10]:
benchmarks = load_benchmark_files(
    directory="benchmarks",
    throughput=16,
    n_partitions=3
)

for b in benchmarks:
    print(
        b["path"],
        b["n_scans_per_batch"],
        b["n_partitions"],
        b["n_workers"]
    )

benchmarks\2026-07-14_15-58-17--16MBps--512SpB--3ws--3parts.json 512 3 3


# Time vs n_workers plot

In [23]:
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np


output_dir = Path("results/times_vs_nworkers")
output_dir.mkdir(parents=True, exist_ok=True)


# Load all valid benchmark files
benchmarks = load_benchmark_files(
    directory="benchmarks",
    verify_content=True,
)


# Group by throughput, partitions, workers
grouped = defaultdict(list)

for b in benchmarks:
    throughput = b["throughput_MB"]
    partitions = b["n_partitions"]
    workers = b["n_workers"]
    analysis_end_ts = b["data"]["analysis_end_ts"]
    producer_begin_ts = b["data"]["producer_begin_ts"]

    grouped[(throughput, partitions, workers)].append(
        {
            "total_time": analysis_end_ts - producer_begin_ts,
            "file": b["path"],
        }
    )
# Create plots
plots = defaultdict(list)

for (throughput, partitions, workers), runs in grouped.items():

    total_times = [r["total_time"] for r in runs]

    plots[(throughput, partitions)].append(
        {
            "workers": workers,
            "mean": np.mean(total_times),
            "std": np.std(total_times),
            "files": [r["file"] for r in runs],
        }
    )


for (throughput, partitions), points in plots.items():

    points.sort(key=lambda x: x["workers"])

    workers = [p["workers"] for p in points]
    means = [p["mean"] for p in points]
    errors = [p["std"] for p in points]


    # Output folder:
    # times_vs_nworkers/<throughput>/<partitions>
    folder = (
        output_dir
        / f"{throughput}MBps"
        / f"{partitions}parts"
    )

    folder.mkdir(parents=True, exist_ok=True)


    # Plot
        # Plot histogram/bar chart
    plt.figure(figsize=(7, 5))

    bars = plt.bar(
        workers,
        means,
        yerr=errors,
        capsize=5,
        align="center",
    )

    plt.xlabel("n_workers")
    plt.ylabel("average total_time")
    plt.title(
        f"Total time vs n_workers\n"
        f"{throughput} MBps, {partitions} partitions"
    )

    # Force integer x-axis ticks
    plt.xticks([1, 2, 3, 4])
    plt.xlim(0.5, 4.5)
    plt.ylim(0, 240)
    plt.yticks([30, 60, 90, 120, 150, 180, 210, 240])

    # Write number of datapoints inside each bar
    for bar, p in zip(bars, points):
        height = bar.get_height()

        n_points = len(p["files"])

        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height / 2,
            str(n_points),
            ha="center",
            va="center",
        )

    plt.gca().set_axisbelow(True)
    plt.grid(
        axis="y",
        #linestyle="--",
        alpha=0.3,
    )
    plt.axhline(120, linestyle="--", color='black')
    plt.tight_layout()

    plt.savefig(
        folder / "total_time_vs_nworkers.png",
        dpi=300,
    )

    plt.close()


    # Save contributing files
    with (folder / "files.txt").open("w") as f:

        f.write(
            f"throughput_MB={throughput}\n"
            f"n_partitions={partitions}\n\n"
        )

        for p in points:
            f.write(
                f"n_workers={p['workers']}\n"
                f"mean_total_time={p['mean']}\n"
                f"std_total_time={p['std']}\n"
            )

            for filename in p["files"]:
                f.write(f"    {filename}\n")

            f.write("\n")

# Time vs n_partitions

In [30]:
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np


def plot_total_time_vs(variable, benchmark_dir="benchmarks", output_dir="results"):
    assert variable in {"throughput", "n_scans", "n_partitions", "n_workers"}

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    x_ticks = {
        "n_workers": [1, 2, 3, 4],
        "n_partitions": list(range(1, 11)),
        "throughput": [16, 20, 24],
        "n_scans": [512, 1024, 2048, 4096],
    }

    benchmarks = load_benchmark_files(directory=benchmark_dir, verify_content=True)

    get_value = {
        "throughput": lambda b: b["throughput_MB"],
        "n_scans": lambda b: b["data"]["n_scans_per_batch"],
        "n_partitions": lambda b: b["n_partitions"],
        "n_workers": lambda b: b["n_workers"],
    }

    fixed_variables = [v for v in get_value if v != variable]

    grouped = defaultdict(list)

    for b in benchmarks:
        x = get_value[variable](b)
        fixed = tuple(get_value[v](b) for v in fixed_variables)

        total_time = b["data"]["analysis_end_ts"] - b["data"]["producer_begin_ts"]

        grouped[(fixed, x)].append({"total_time": total_time, "file": b["path"]})

    plots = defaultdict(list)

    for (fixed, x), runs in grouped.items():
        times = [r["total_time"] for r in runs]
        plots[fixed].append({
            "x": x,
            "mean": np.mean(times),
            "std": np.std(times),
            "files": [r["file"] for r in runs],
        })

    short_names = {"throughput": "MBps", "n_scans": "SpB", "n_partitions": "parts", "n_workers": "ws"}

    for fixed, points in plots.items():
        points.sort(key=lambda p: p["x"])

        xs = [p["x"] for p in points]
        means = [p["mean"] for p in points]
        errors = [p["std"] for p in points]

        fixed_dict = dict(zip(fixed_variables, fixed))

        fixed_string = "__".join(f"{k}{v}{short_names[k]}" for k, v in fixed_dict.items())
        filename = f"time_vs_{variable}__{fixed_string}.png"

        folder = output_dir
        folder.mkdir(parents=True, exist_ok=True)

        plt.figure(figsize=(7, 5))

        bars = plt.bar(xs, means, yerr=errors, capsize=5, align="center")

        plt.xlabel(variable)
        plt.ylabel("average tot time")

        plt.title(f"Total time vs {variable}\n" + " ".join(f"{k}={v}" for k, v in fixed_dict.items()))

        plt.xticks(x_ticks[variable])
        plt.xlim(min(x_ticks[variable]) - 0.5, max(x_ticks[variable]) + 0.5)

        throughput = fixed_dict.get("throughput")
        if throughput is None:
            raise ValueError("throughput must be fixed for time plots")

        max_y = 3840 / throughput

        plt.ylim(0, max_y)
        plt.yticks(np.arange(0, max_y + 30, 30))

        for bar, p in zip(bars, points):
            plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2, str(len(p["files"])), ha="center", va="center")

        plt.gca().set_axisbelow(True)
        plt.grid(axis="y", alpha=0.3)
        plt.axhline(1920 / throughput, linestyle="--", color="black")

        plt.tight_layout()
        plt.savefig(folder / filename, dpi=300)
        plt.close()

        txt_filename = filename.replace(".png", ".txt")

        with (folder / txt_filename).open("w") as f:
            f.write(f"variable={variable}\n")
            for k, v in fixed_dict.items():
                f.write(f"{k}={v}\n")
            f.write("\n")

            for p in points:
                f.write(f"{variable}={p['x']}\n")
                f.write(f"mean_total_time={p['mean']}\n")
                f.write(f"std_total_time={p['std']}\n")
                for filename in p["files"]:
                    f.write(f"    {filename}\n")
                f.write("\n")

In [31]:
plot_total_time_vs("n_workers")
plot_total_time_vs("n_partitions")